# AAL Sales Analysis · Q4 2020

**Applied Data Science with Python | Course-end project**  
**Working style:** follow the assignment's four sections in order. We complete **Sections 1 and 2** before beginning Section 3.

| Rubric section | Status |
| --- | --- |
| **1. Data Wrangling** (a–d) | Complete |
| **2. Data Analysis** (a–d) | Ready to run |
| **3. Data Visualization** | To be completed |
| **4. Report Generation** | To be completed |

The data describes apparel sales in Australia. Its source does **not** specify a currency, so we refer to the `Sales` column as recorded sales values without assuming AUD.

**Before running:** place the supplied `AusApparalSales4thQrt2020.csv` beside this notebook in Practice Labs. If working from the downloaded project folder, leave the original ZIP in the `data/` directory. Both arrangements are supported.

## 1. Data Wrangling

**Rubric goal:** (a) check missing/incorrect records, (b) justify how to handle them, (c) apply normalization, and (d) demonstrate and explain `GroupBy()`.

### 1a. Inspect missing and incorrect entries

**Step 1: Load the original dataset.** `pandas` (usually imported as `pd`) reads the CSV into a *DataFrame*: a table with rows and columns. This cell also accepts the original ZIP if you are working from the project folder.

In [ ]:
from pathlib import Path
import pandas as pd

# The first existing file wins. Practice Labs usually keeps CSV and notebook together.
candidates = [
    Path('AusApparalSales4thQrt2020.csv'),
    Path('data/AusApparalSales4thQrt2020.csv'),
    Path('../data/AusApparalSales4thQrt2020.csv'),
    Path('1739523823_ausapparalsales4thqrt2020.zip'),
    Path('data/1739523823_ausapparalsales4thqrt2020.zip'),
    Path('../data/1739523823_ausapparalsales4thqrt2020.zip'),
]
source_file = next((path for path in candidates if path.is_file()), None)
if source_file is None:
    raise FileNotFoundError('Put the CSV beside this notebook, or the original ZIP in data/.')

raw_df = pd.read_csv(source_file)
print('Source file:', source_file)
print(f'Rows: {len(raw_df):,}; columns: {len(raw_df.columns)}')
display(raw_df.head())

**Step 2: Check data types and missing cells.** `isna()` gives `True` for missing values; `notna()` does the opposite. Summing Boolean values counts them.

In [ ]:
print('Original data types:')
display(raw_df.dtypes.to_frame('dtype'))

missing_check = pd.DataFrame({
    'missing: isna()': raw_df.isna().sum(),
    'present: notna()': raw_df.notna().sum(),
})
display(missing_check)

**Step 3: Clean a copy, not the source.** The text fields have leading spaces, so we remove them with `.str.strip()`. We convert dates and numeric fields explicitly. `errors='coerce'` turns an invalid entry into a detectable missing value rather than hiding a parsing error.

In [ ]:
df = raw_df.copy()  # The original raw_df stays unchanged.
df.columns = df.columns.str.strip()

for column in ['Time', 'State', 'Group']:
    df[column] = df[column].astype('string').str.strip()

df['Date'] = pd.to_datetime(
    df['Date'].astype('string').str.strip(), format='%d-%b-%Y', errors='coerce'
)
for column in ['Unit', 'Sales']:
    df[column] = pd.to_numeric(df[column], errors='coerce')

print('Example of removing extra spaces:')
print('Before:', repr(raw_df.loc[0, 'Group']))
print('After: ', repr(df.loc[0, 'Group']))

display(df.head())

**Step 4: Inspect potential errors after cleaning.** A *duplicate business key* would mean more than one entry for the same Date × Time × State × Group. Negative sales might be returns in another business, so we flag them for investigation rather than deleting them automatically.

In [ ]:
print('Missing or invalid values after conversion:')
display(df.isna().sum().to_frame('Count'))

check_keys = ['Date', 'Time', 'State', 'Group']
quarter_start = pd.Timestamp('2020-10-01')
quarter_end = pd.Timestamp('2020-12-31')

issues = pd.Series({
    'Identical duplicate rows': int(df.duplicated().sum()),
    'Duplicate date/time/state/group keys': int(df.duplicated(check_keys).sum()),
    'Out-of-quarter dates': int((df['Date'].notna() & ~df['Date'].between(quarter_start, quarter_end)).sum()),
    'Units <= 0': int((df['Unit'] <= 0).sum()),
    'Fractional units': int(((df['Unit'] % 1 != 0) & df['Unit'].notna()).sum()),
    'Negative sales values': int((df['Sales'] < 0).sum()),
    'Unexpected group labels': int((df['Group'].notna() & ~df['Group'].isin(['Kids', 'Men', 'Women', 'Seniors'])).sum()),
    'Unexpected time labels': int((df['Time'].notna() & ~df['Time'].isin(['Morning', 'Afternoon', 'Evening'])).sum()),
})
display(issues.to_frame('Rows flagged'))

for column in ['State', 'Group', 'Time']:
    print(column, ':', sorted(df[column].dropna().unique().tolist()))

**Step 5: Check whether any *whole days* are absent.** A dataset can have zero `NaN` cells while missing entire days of trading. Q4 2020 has **92 calendar days**, so we compare that calendar with the distinct dates in the file.

In [ ]:
quarter_dates = pd.date_range('2020-10-01', '2020-12-31', freq='D')
observed_dates = pd.DatetimeIndex(df['Date'].dropna().unique())
missing_dates = quarter_dates.difference(observed_dates)

print('Recorded calendar days:', len(observed_dates), '/', len(quarter_dates))
print('Unrecorded dates:')
for day in missing_dates:
    print(' ', day.strftime('%d %b %Y'))

rows_each_day = df.groupby('Date').size()
print('Rows per recorded day: minimum', rows_each_day.min(), 'maximum', rows_each_day.max())

### 1b. Recommend how to treat missing or incorrect information

**Decision for this dataset:**

| Finding | Recommended treatment | Reason |
| --- | --- | --- |
| Missing values in existing rows | **No filling or dropping required** if the checks above show zero | We should not change valid data without a reason. |
| Extra spaces in text fields | **Strip whitespace** | Otherwise `' Men'` and `'Men'` could be treated as different categories. |
| Incorrect dates, amounts or duplicate keys (if found later) | **Investigate against the source before dropping or filling** | Automatic fixes can create false business results. |
| Entire dates absent (31 October and 31 December 2020) | **Keep them identified as unrecorded; do not fill with zero or invented averages** | A missing day does not prove the business earned no sales. |

**Hypothesis, unverified:** 31 October is Halloween and 31 December is New Year's Eve. Holiday-related closures or altered trading *could* explain the two gaps, but a reporting or extraction omission is also possible. The file contains no opening-hours data. Halloween is not generally a day when Australian retailers close, so neither date should be labelled a confirmed closure without evidence.

**Why not `dropna()` or `fillna()` here?** Those functions operate on *existing rows*, whereas two entire days are missing from the file. We would need independent trading schedules or original transaction records to resolve that question.

### 1c. Apply Min-Max normalization

**Chosen method: Min-Max normalization**, as recommended by the assignment. It rescales each numeric column to a value between **0 and 1**:

\[
 x_{\mathrm{normalized}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}
\]

We will normalize **`Unit`** and **`Sales`**, leaving the originals intact. We should **not** normalize dates or category labels.

In [ ]:
for column in ['Unit', 'Sales']:
    minimum = df[column].min()
    maximum = df[column].max()

    if maximum == minimum:
        raise ValueError(f'Cannot apply Min-Max to {column}: all values are identical.')

    df[f'{column}_normalized'] = (df[column] - minimum) / (maximum - minimum)

print('Original and normalized values:')
display(df[['Unit', 'Unit_normalized', 'Sales', 'Sales_normalized']].head(8))

print('Normalized minimum and maximum (both must be 0 and 1):')
display(df[['Unit_normalized', 'Sales_normalized']].agg(['min', 'max']))

**Why might the two normalized columns be identical?** Inspect the ratio `Sales / Unit`. In this particular dataset, each sale value equals **2,500 × units**, so both columns carry the same relative information. Normalization does **not** make them independent.

In [ ]:
ratio_values = sorted((df['Sales'] / df['Unit']).dropna().unique().tolist())
print('Distinct Sales / Unit ratios:', ratio_values)
print('Greatest difference between normalized columns:',
      (df['Sales_normalized'] - df['Unit_normalized']).abs().max())

### 1d. Apply `GroupBy()` and recommend its use

`groupby()` uses **split → apply → combine**: split rows into logical groups, calculate a value per group, then combine the results into one summary table.

For AAL, **grouping (data chunking)** is more useful than merging at this stage. We already have all columns in one table, and we need state-wise and demographic summaries. A **merge** would be appropriate later if we obtained a *different table*, such as store locations or marketing spend.

**Important:** aggregate the original `Sales` and `Unit` values, not the normalized columns. Sums of normalized scores are not meaningful revenue totals.

In [ ]:
by_state = (
    df.groupby('State', as_index=False)
      .agg(Records=('Sales', 'size'), Total_Units=('Unit', 'sum'), Total_Sales=('Sales', 'sum'))
      .sort_values('Total_Sales', ascending=False)
)

by_state_and_group = (
    df.groupby(['State', 'Group'], as_index=False)
      .agg(Records=('Sales', 'size'), Total_Sales=('Sales', 'sum'))
      .sort_values(['State', 'Group'])
)

print('State-level groupings:')
display(by_state)
print('First ten state-by-group combinations:')
display(by_state_and_group.head(10))

**Section 1 verification.** The following quick checks are meant to catch mistakes before we begin the statistical analysis. They do not change any data.

In [ ]:
assert len(raw_df) == len(df), 'Rows must not disappear during cleaning.'
assert df[['Unit_normalized', 'Sales_normalized']].notna().all().all()
assert df['Unit_normalized'].between(0, 1).all()
assert df['Sales_normalized'].between(0, 1).all()
assert abs(by_state['Total_Sales'].sum() - df['Sales'].sum()) < 0.000001
assert (df['Unit_normalized'] - df['Sales_normalized']).abs().max() < 0.000000001
print('Section 1 checks passed: row count preserved, normalized values in [0,1], grouped sales reconciled.')

**Section 1 recap.** We checked data quality, documented the missing-date treatment, normalized without changing source values, and practised `GroupBy()`. Now we can analyse the *original* Sales and Unit columns.

---
## 2. Data Analysis

**Rubric goal:** (a) descriptive statistics for Unit and Sales, (b) identify highest/lowest selling demographic groups, (c) compare the highest/lowest selling states, and (d) prepare weekly, monthly and quarterly reports.

The original assignment repeats the group question in 2b and 2c. We use 2c to answer the wider project question about states, without ignoring the duplicate wording.

### 2a. Calculate descriptive statistics

We will find **mean**, **median**, **mode** and **standard deviation** for both numeric columns:

- **Mean:** arithmetic average (sum divided by number of rows).
- **Median:** the middle value when data is ordered.
- **Mode:** the value appearing most often.
- **Standard deviation:** how much values vary around the mean. Pandas `std()` uses the *sample* standard deviation (`ddof=1`).

Run this cell **after Section 1**, because it uses our cleaned `df`.

In [ ]:
# Calculate the summary for each original numeric column.
statistics = df[['Unit', 'Sales']].agg(['mean', 'median', 'std']).T
statistics['mode'] = [df[column].mode().iloc[0] for column in ['Unit', 'Sales']]
statistics = statistics[['mean', 'median', 'mode', 'std']]

print('Descriptive statistics for each recorded row:')
display(statistics.round(2))
print(f"Sales mode: {statistics.loc['Sales', 'mode']:,.0f}")

**What did we learn?** The recorded mean sale is about **45,013.56**, the median is **35,000**, and the most common sale value is **22,500** (currency unspecified). The mean being above the median suggests larger transactions pull the average upward. These are values **per dataset row**, not per individual shop or customer; the dataset does not identify transactions.

### 2b. Find the highest and lowest selling demographic groups

We will group the original **Sales** and **Unit** fields by `Group`. We use the *total* sales value to rank Kids, Men, Women and Seniors. Counts are included so we can see whether groups have similar numbers of records.

In [ ]:
group_report = (
    df.groupby('Group', as_index=False)
      .agg(Records=('Sales', 'size'), Total_Units=('Unit', 'sum'), Total_Sales=('Sales', 'sum'))
      .sort_values('Total_Sales', ascending=False)
      .reset_index(drop=True)
)

display(group_report)
print('Highest selling group:', group_report.iloc[0]['Group'])
print('Lowest selling group: ', group_report.iloc[-1]['Group'])

**Observation:** Men have the highest recorded total (**85,750,000**), and Seniors the lowest (**84,037,500**). Each demographic group contributes **1,890** records. The gap is modest compared with the overall size of the dataset; these totals alone do not explain *why* a group leads.

### 2c. Find the highest and lowest selling states or territories

The assignment's 2b and 2c repeat the group question, but the project statement explicitly asks for **state-by-state sales**. We address this here, ranking states by the sum of original Sales.

In [ ]:
state_report = (
    df.groupby('State', as_index=False)
      .agg(Records=('Sales', 'size'), Total_Units=('Unit', 'sum'), Total_Sales=('Sales', 'sum'))
      .sort_values('Total_Sales', ascending=False)
      .reset_index(drop=True)
)

display(state_report)
print('Highest selling state:', state_report.iloc[0]['State'])
print('Lowest selling state: ', state_report.iloc[-1]['State'])

**Observation:** Victoria (VIC) is highest at **105,565,000** and Western Australia (WA) is lowest at **22,152,500**. Every recorded state/territory has **1,080** rows. We cannot infer store productivity or market demand without store counts, populations, or other context. The source does not include ACT.

### 2d. Produce weekly, monthly and quarterly reports

A **time-based GroupBy** creates reports at different levels of detail. We first convert each date to a weekly or monthly **period**; this labels which calendar bucket each row belongs to. We sum original sales and units and count distinct **recorded** days. No missing days are filled or estimated.

**Weekly:** We define weeks as **Monday to Sunday**, using `W-SUN`. The first and last weeks extend beyond Q4, so their totals only include Q4 records from the source; those rows are not comparable with complete weeks without adjusting for day counts.

In [ ]:
weekly_report = (
    df.assign(Week=df['Date'].dt.to_period('W-SUN'))
      .groupby('Week', as_index=False)
      .agg(Recorded_Days=('Date', 'nunique'), Total_Units=('Unit', 'sum'),
           Total_Sales=('Sales', 'sum'))
      .sort_values('Week')
)

print('Weekly sales report (Monday-Sunday calendar weeks):')
display(weekly_report)
print('Number of weekly reporting periods:', len(weekly_report))

**Monthly:** Converting to a monthly period with `M` produces one total for **October, November and December**. `Recorded_Days` shows why two months have incomplete calendar coverage: October and December each have only 30 recorded days, and November has 30 too because it has 30 calendar days.

In [ ]:
monthly_report = (
    df.assign(Month=df['Date'].dt.to_period('M'))
      .groupby('Month', as_index=False)
      .agg(Recorded_Days=('Date', 'nunique'), Total_Units=('Unit', 'sum'),
           Total_Sales=('Sales', 'sum'))
      .sort_values('Month')
)

print('Monthly sales report:')
display(monthly_report)

**Quarterly:** The `Q` period groups the entire quarter together. The quarter contains **90 recorded dates**, although October through December have **92 calendar dates**. We report recorded values only and flag the two missing dates as unverified gaps.

In [ ]:
quarterly_report = (
    df.assign(Quarter=df['Date'].dt.to_period('Q'))
      .groupby('Quarter', as_index=False)
      .agg(Recorded_Days=('Date', 'nunique'), Total_Units=('Unit', 'sum'),
           Total_Sales=('Sales', 'sum'))
      .sort_values('Quarter')
)

print('Quarterly sales report:')
display(quarterly_report)
print('Quarter total recorded sales:', f"{quarterly_report['Total_Sales'].sum():,.0f}")
print('Quarter labels:', quarterly_report['Quarter'].astype(str).tolist())

**What do the time summaries suggest?** November has the lowest recorded monthly sales (**90,682,500**), while December has the highest (**135,330,000**). A seasonal explanation is *possible*, but one Q4 dataset does not demonstrate a recurring trend. We should avoid treating Halloween or New Year's Eve as confirmed closures.

### Section 2 verification

These checks make sure the totals reconcile across the demographic, geographic and time-based reports. The tables must all add back to the original source's recorded sales and units; none should create or discard data.

In [ ]:
for report in [group_report, state_report, weekly_report, monthly_report, quarterly_report]:
    assert report['Total_Sales'].sum() == df['Sales'].sum()
    assert report['Total_Units'].sum() == df['Unit'].sum()

assert weekly_report['Recorded_Days'].sum() == df['Date'].nunique()
assert monthly_report['Recorded_Days'].sum() == df['Date'].nunique()
assert quarterly_report['Recorded_Days'].sum() == df['Date'].nunique()

print('Section 2 checks passed: all group, state and period totals reconcile.')

---
## 3. Data Visualization

**To be completed after Section 2.** We will create appropriate Seaborn/Matplotlib plots for sales by state and demographic, time of day, and daily/weekly/monthly/quarterly trends. We will also explain our choice of plotting library.

---
## 4. Report Generation

**To be completed last.** We will bring the methods, statistical findings, box plots, distributions, limitations and evidence-based business recommendations together into a JupyterLab report.